# Fine-tuning an LLM with LoRA (SFT with TRL + PEFT)

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/05-LoRA-SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers peft trl
!pip uninstall -q -y torchao   # the old torchao preinstalled on Colab is incompatible with recent PEFT
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['zh_numbers.py', 'q_lora_params.py', 'q_upper.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

Fine-tuning all the parameters of a large model takes a lot of GPU memory (parameters + gradients + optimizer state, roughly 4x the model size or more).

**LoRA (Low-Rank Adaptation)**: freeze the original weights $W$ and train only a low-rank correction

$$W' = W + \frac{\alpha}{r} BA, \quad B\in\mathbb{R}^{d\times r}, A\in\mathbb{R}^{r\times k}, r \ll d$$

The trainable parameters are usually less than 1% of the model, so fine-tuning fits even on a T4.

The task in this unit: teach Qwen3-0.6B to **convert Chinese numerals into Arabic digits** (e.g. 三千零五十二 → 3052).
It looks easy, but small models often get it wrong. The data is generated by code (`zh_numbers.py`), so we can make as much as we want, and accuracy is easy to measure.

**Chinese numerals in a nutshell**
* Digits: 一 1, 二 2, 三 3, 四 4, 五 5, 六 6, 七 7, 八 8, 九 9 (零 is 0)
* Units: 十 ten, 百 hundred, 千 thousand, 萬 ten-thousand (億 is 10⁸). A number is read as digit + unit pairs: 三千零五十二 = 3×1000 + 5×10 + 2 = 3052. Large numbers group by 萬, not by thousand: 一萬零五十 = 10050.
* Zero rule: a run of missing places inside a number is written as a single 零 placeholder (3052 → 三千**零**五十二, 10100100 → 一千零一十萬零一百); trailing zeros are not written (20000000 → 二千萬).
* Formal ("banker's") numerals: on checks and financial documents, each character is replaced by a more complex form — 壹 貳 參 肆 伍 陸 柒 捌 玖 for 1–9, 拾 佰 仟 for ten/hundred/thousand — because simple strokes like 一 or 二 are easy to alter into a bigger number. 3052 → 參仟零伍拾貳.

In [ ]:
import random, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from zh_numbers import num2zh, num2zh_upper, random_number
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
for n in [7, 15, 105, 3052, 10050, 20000000, 10100100]:
    print(n, num2zh(n))

## Preparing the data

TRL's `SFTTrainer` supports conversational data in prompt-completion format, and by default computes the loss only on the completion (the model's answer).

In [ ]:
from datasets import Dataset
PROMPT = "Convert the following Chinese numeral to Arabic digits. Output only the number: {}"

def make_example(n, to_zh=num2zh):
    return {"prompt": [{"role": "user", "content": PROMPT.format(to_zh(n))}],
            "completion": [{"role": "assistant", "content": str(n)}],
            "answer": n}

rng = random.Random(42)
test_numbers = [random_number(rng) for _ in range(200)]
train_numbers = [n for n in (random_number(rng) for _ in range(6000)) if n not in set(test_numbers)][:4000]
train_ds = Dataset.from_list([make_example(n) for n in train_numbers])
test_ds = Dataset.from_list([make_example(n) for n in test_numbers])
train_ds[0]

## Before training: how does the original model do?

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

In [ ]:
@torch.no_grad()
def evaluate(model, ds, batch_size=50, show=5):
    tokenizer.padding_side = "left"
    model.eval()
    correct, outputs = 0, []
    for i in range(0, len(ds), batch_size):
        batch = ds[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                 for p in batch["prompt"]]
        inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        for o, ans, p in zip(out, batch["answer"], batch["prompt"]):
            pred = tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().replace(",", "")
            correct += pred == str(ans)
            outputs.append((p[0]["content"].split(": ")[-1], ans, pred))
    for question, ans, pred in outputs[:show]:
        print(f"{question} → {pred}  {'✅' if pred == str(ans) else '❌ ' + str(ans)}")
    return correct / len(ds)

print("accuracy:", evaluate(model, test_ds))

## Adding LoRA

`target_modules="all-linear"`: add LoRA to every linear layer (attention q, k, v, o and the MLP).

In [ ]:
from peft import LoraConfig
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

## Training with TRL's SFTTrainer

Notes for the T4:
* The model is loaded in fp16, so use `fp16=True`, `bf16=False` (SFTConfig turns on bf16 by default, which the T4 does not support)
* The LoRA parameters are trained in fp32 automatically, together with a GradScaler, which is numerically more stable

In [ ]:
from trl import SFTConfig, SFTTrainer
tokenizer.padding_side = "right"
args = SFTConfig(
    output_dir="qwen3-zh-number-lora",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=20,
    max_length=128,
    fp16=True,
    bf16=False,
    save_strategy="no",
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds.remove_columns("answer"),
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
# Look at what is actually fed to the model: tokens whose label is -100 are excluded from the loss (only the model's answer counts)
ex = trainer.train_dataset[0]
print(list(ex.keys()))
print(tokenizer.decode(ex["input_ids"]))
if "labels" in ex:
    print("part used for the loss:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

In [ ]:
t0 = time.perf_counter()
trainer.train()
print(f"training time {time.perf_counter() - t0:.0f}s")

## After training

In [ ]:
print("accuracy:", evaluate(trainer.model, test_ds, show=10))

## Q
What fraction of the parameters does LoRA actually train? Count the parameters in `trainer.model` with `requires_grad=True` yourself, and list the names and shapes of the first few.

In [ ]:
# %load q_lora_params.py
%run -i q_lora_params.py

## Saving and merging

A LoRA adapter is small (a few MB to a few tens of MB), so it can be saved and shared on its own. You can also merge $BA$ back into the original weights to get an ordinary model.

In [ ]:
trainer.save_model("qwen3-zh-number-lora")       # saves only the adapter
!ls -lh qwen3-zh-number-lora

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("qwen3-zh-number-merged")
tokenizer.save_pretrained("qwen3-zh-number-merged")

In [ ]:
# The merged model is a plain transformers model. Does it keep its other abilities?
messages = [{"role": "user", "content": "Explain what Python is in one sentence."}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                       return_dict=True, return_tensors="pt").to(device)
out = merged.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

## Q
So far the model has only seen ordinary Chinese numerals. Use `num2zh_upper` (formal numerals, e.g. 參仟零伍拾貳) to build a test set:
1. What is the current model's accuracy on formal numerals?
2. Add formal-numeral data to the training set and train a new LoRA. Can it convert both forms correctly?

In [ ]:
# %load q_upper.py
%run -i q_upper.py